In [2]:
!nvidia-smi

Sun Oct 25 20:23:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!df -h /kaggle/working /kaggle/tmp / 2>/dev/null
!nproc; free -g

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   72K   20G   1% /kaggle/working
overlay         8.0T  7.0T  1.1T  88% /
4
               total        used        free      shared  buff/cache   available
Mem:              31           1          25           0           5          29
Swap:              0           0           0


In [4]:
!git clone --depth 1 --branch v0.7.0 https://github.com/ggml-org/llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 4155, done.
remote: Counting objects: 100% (4155/4155), done.
remote: Compressing objects: 100% (3426/3426), done.
remote: Total 4155 (delta 755), reused 2861 (delta 647), pack-reused 0 (from 0)
Receiving objects: 100% (4155/4155), 36.73 MiB | 17.25 MiB/s, done.
Resolving deltas: 100% (755/755), done.
Note: switching to '620d2225cc436ab0b001e3200e590e3b286ef9ab'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false



In [5]:
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON \
    -DCMAKE_CUDA_ARCHITECTURES=75 -DGGML_CUDA_NO_VMM=ON -DLLAMA_BUILD_TESTS=OFF

-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.7.0-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.43.0")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found Ope

In [6]:
!cmake --build llama.cpp/build --config Release -j4 \
    --target llama-quantize llama-completion llama-perplexity llama-server
!ls llama.cpp/build/bin/

[  0%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml.c.o
[  0%] Building CXX object common/CMakeFiles/llama-common-base.dir/build-info.cpp.o
[  0%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml.cpp.o
[  0%] Building CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o
[  0%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-alloc.c.o
[  0%] Linking CXX static library libllama-common-base.a
[  0%] Built target llama-common-base
[  0%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend.cpp.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend-meta.cpp.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-opt.cpp.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-threading.cpp.o
[  1%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-quants.c.o
[  1%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/gguf.cpp.o
[  3%] Linking CXX shared library ../../bin/libggm

In [9]:
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!pip install -q "huggingface_hub<1.0"
!python -c "import transformers, huggingface_hub; print('transformers', transformers.__version__, '| hub', huggingface_hub.__version__)"
!hf --help | head -5

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 4.57.6 | hub 0.36.2
usage: hf <command> [<args>]

positional arguments:
  {auth,cache,download,jobs,repo,repo-files,upload,upload-large-folder,env,version,lfs-enable-largefiles,lfs-multipart-upload}
                        hf command helpers


In [8]:
!hf download allenai/OLMoE-1B-7B-0924 --local-dir $MODELS/olmoe-hf

Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 12 files:   0%|                                 | 0/12 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0%|        |  0.00B /   120B            

Fetching 12 files:   8%|██                       | 1/12 [00:00<00:01,  9.90it/s]Still waiting to acquire lock on /olmoe-hf/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /olmoe-hf/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)

Reconstructing (incomplete total...):   0%|        |   120B /  287kB            
Reconstructing (incomplete total...):  98%|███████▊|  287kB /  295kB            
Reconstructing (incomplete total...):   0%|        |  295

In [10]:
!python -c "import transformers, huggingface_hub; print('transformers', transformers.__version__, '| hub', huggingface_hub.__version__)"
!hf --help | head -5

transformers 4.57.6 | hub 0.36.2
usage: hf <command> [<args>]

positional arguments:
  {auth,cache,download,jobs,repo,repo-files,upload,upload-large-folder,env,version,lfs-enable-largefiles,lfs-multipart-upload}
                        hf command helpers


In [11]:
!python llama.cpp/convert_hf_to_gguf.py $MODELS/olmoe-hf --outtype bf16 --outfile $MODELS/olmoe-bf16.gguf
!ls -lh $MODELS

INFO:hf-to-gguf:Loading model: olmoe-hf
INFO:hf-to-gguf:Model architecture: OlmoeForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00003.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00003.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00003.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> BF16, shape = {2048, 50304}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down_exps.weight,  torch.bfloat16 --> BF16, shape = {1024, 2048, 64}
INFO:hf-to-gguf:blk.0.ffn_gate_exps.weight,  torch.bfloat16 --> BF16, shape = {2048, 1024, 64}
INFO:hf-to-gguf:blk.0.ffn_up_exps.weight,    torch.bfloat16 --> BF16, shape = {2048, 1024, 64}
INFO:hf-to-gguf:blk.0.ffn_ga

In [12]:
!rm -rf $MODELS/olmoe-hf
!df -h $MODELS

Filesystem        Size  Used Avail Use% Mounted on
overlay           8.0T  7.0T  1.1T  88% /
tmpfs              64M     0   64M   0% /dev
shm                14G     0   14G   0% /dev/shm
/dev/sda1         122G   32G   91G  26% /opt/bin
/dev/loop2         20G  484M   20G   3% /kaggle/lib
/dev/mapper/snap  8.0T  7.0T  1.1T  88% /etc/hosts
tmpfs             4.0K     0  4.0K   0% /proc/acpi


In [13]:
%%bash
for q in Q8_0 Q6_K Q5_K_M Q4_K_M Q3_K_M Q2_K; do
  echo "=== $q"
  llama.cpp/build/bin/llama-quantize --output-tensor-type q8_0 \
    $MODELS/olmoe-bf16.gguf $MODELS/olmoe-$q.gguf $q 2>&1 | tail -3
done
ls -lh $MODELS

=== Q8_0

llama_quantize: quantize time = 77770.40 ms
llama_quantize:    total time = 77770.40 ms
=== Q6_K

llama_quantize: quantize time = 191687.83 ms
llama_quantize:    total time = 191687.83 ms
=== Q5_K_M

llama_quantize: quantize time = 300702.35 ms
llama_quantize:    total time = 300702.35 ms
=== Q4_K_M

llama_quantize: quantize time = 340276.30 ms
llama_quantize:    total time = 340276.30 ms
=== Q3_K_M

llama_quantize: quantize time = 160140.61 ms
llama_quantize:    total time = 160140.61 ms
=== Q2_K

llama_quantize: quantize time = 208403.87 ms
llama_quantize:    total time = 208403.87 ms
total 4.0K
drwxr-xr-x 31 root root 4.0K Oct 25 20:24 llama.cpp


In [14]:
!ls -lh /olmoe-* /kaggle/tmp/models/ 2>&1

ls: cannot access '/kaggle/tmp/models/': No such file or directory
-rw-r--r-- 1 root root  13G Oct 25 21:00 /olmoe-bf16.gguf
-rw-r--r-- 1 root root 2.5G Oct 25 21:23 /olmoe-Q2_K.gguf
-rw-r--r-- 1 root root 3.2G Oct 25 21:19 /olmoe-Q3_K_M.gguf
-rw-r--r-- 1 root root 4.0G Oct 25 21:16 /olmoe-Q4_K_M.gguf
-rw-r--r-- 1 root root 4.7G Oct 25 21:11 /olmoe-Q5_K_M.gguf
-rw-r--r-- 1 root root 5.4G Oct 25 21:06 /olmoe-Q6_K.gguf
-rw-r--r-- 1 root root 6.9G Oct 25 21:02 /olmoe-Q8_0.gguf


In [15]:
!mkdir -p /kaggle/tmp/models && mv /olmoe-*.gguf /kaggle/tmp/models/ && ls -l /kaggle/tmp/models/
!python llama.cpp/gguf-py/gguf/scripts/gguf_dump.py /kaggle/tmp/models/olmoe-Q2_K.gguf | grep -E "gate_inp|output.weight" | head

total 41075280
-rw-r--r-- 1 root root 13844565856 Oct 25 21:00 olmoe-bf16.gguf
-rw-r--r-- 1 root root  2587713376 Oct 25 21:23 olmoe-Q2_K.gguf
-rw-r--r-- 1 root root  3368879968 Oct 25 21:19 olmoe-Q3_K_M.gguf
-rw-r--r-- 1 root root  4238462816 Oct 25 21:16 olmoe-Q4_K_M.gguf
-rw-r--r-- 1 root root  4951789408 Oct 25 21:11 olmoe-Q5_K_M.gguf
-rw-r--r-- 1 root root  5709698912 Oct 25 21:06 olmoe-Q6_K.gguf
-rw-r--r-- 1 root root  7359942496 Oct 25 21:02 olmoe-Q8_0.gguf
INFO:gguf-dump:* Loading: /kaggle/tmp/models/olmoe-Q2_K.gguf
      1:  103022592 |  2048, 50304,     1,     1 | Q8_0    | output.weight
      7:    4194304 |  2048,  2048,     1,     1 | Q3_K    | blk.0.attn_output.weight
     13:     131072 |  2048,    64,     1,     1 | F32     | blk.0.ffn_gate_inp.weight
     19:    4194304 |  2048,  2048,     1,     1 | Q3_K    | blk.1.attn_output.weight
     25:     131072 |  2048,    64,     1,     1 | F32     | blk.1.ffn_gate_inp.weight
     31:    4194304 |  2048,  2048,     1,     1 

In [16]:
!for q in bf16 Q8_0 Q4_K_M Q2_K; do echo "=== $q"; llama.cpp/build/bin/llama-completion -m /kaggle/tmp/models/olmoe-$q.gguf -ngl 99 -no-cnv -p "The capital of France is" -n 20 2>/dev/null; echo; done

=== bf16
The capital of France is Paris, it was founded by the Romans in the 1st century AD. It is located on the


=== Q8_0
The capital of France is Paris. The largest city in France is Paris. The official language spoken in France is French.



=== Q4_K_M
The capital of France is Paris.

The population of France is 64,057,214. The population is growing at


=== Q2_K
The capital of France is the city of Paris. The city of Paris has the most people of all of France. The city




In [17]:
!cd llama.cpp && sh scripts/get-wikitext-2.sh

--2026-10-25 21:27:44--  https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip
Resolving huggingface.co (huggingface.co)... 3.167.112.38, 3.167.112.45, 3.167.112.96, ...
Connecting to huggingface.co (huggingface.co)|3.167.112.38|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65d261003da87ce21e5997f8/b28df100478bded493177f9226b11cd70414e3b60ae217270a1cbdf67fbca0b0?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27wikitext-2-raw-v1.zip%3B+filename%3D%22wikitext-2-raw-v1.zip%22%3B&xip=jWeRKgbRyv0&response-content-type=application%2Fzip&user_id=public&X-Xet-Cas-Uid=public&Expires=1792967264&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjVkMjYxMDAzZGE4N2NlMjFlNTk5N2Y4L2IyOGRmMTAwNDc4YmRlZDQ5MzE3N2Y5MjI2YjExY2Q3MDQxNGUzYjYwYWUyMTcyNzBhMWNiZGY2N2ZiY2EwYjBcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomeGlwPWpXZVJLZ2JSeXYwJnJlc3BvbnNlLWNvbnR

In [18]:
!llama.cpp/build/bin/llama-perplexity -m /kaggle/tmp/models/olmoe-Q4_K_M.gguf -f llama.cpp/wikitext-2-raw/wiki.test.raw -ngl 99 --chunks 100

0.02.250.424 I cmn          init: llama threadpool init, n_threads = 2
0.02.344.927 I 
0.02.345.102 I system_info: n_threads = 2 (n_threads_batch = 2) / 4 | CUDA : ARCHS = 750 | NO_VMM = 1 | USE_GRAPHS = 1 | FA_QUANTS = q4_0-q4_0,q8_0-q8_0,f16-f16,bf16-bf16 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | AVX512 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
0.02.345.130 I perplexity: tokenizing the input ..
0.03.151.122 I perplexity: tokenization took 805.982 ms
0.03.151.432 I perplexity: calculating perplexity over 100 chunks, n_ctx=512, batch_size=2048, n_seq=4
0.04.169.841 I perplexity: 1.02 seconds per pass - ETA 0.42 minutes
[1]6.0994,[2]7.6660,[3]6.8687,[4]6.8888,[5]6.8842,[6]6.8645,[7]7.2501,[8]7.4837,[9]8.0106,[10]8.1945,[11]8.2436,[12]8.3973,[13]8.7879,[14]8.5462,[15]8.5275,[16]8.5610,[17]8.5299,[18]8.8240,[19]8.6521,[20]8.6115,[21]8.6912,[22]8.5133,[23]8.2773,[24]8.2718,[25]8.0757,[26]8.0311,[27]8.1030,[28]8.1249,[29]8.1843,[30]8.2051,[31

In [19]:
!for q in bf16 Q8_0 Q6_K Q5_K_M Q4_K_M Q3_K_M Q2_K; do s=$(date +%s); r=$(llama.cpp/build/bin/llama-perplexity -m /kaggle/tmp/models/olmoe-$q.gguf -f llama.cpp/wikitext-2-raw/wiki.test.raw -ngl 99 2>&1 | grep "Final estimate"); echo "$q | $r | $(( $(date +%s) - s ))s"; done

bf16 | 7.47.573.202 I Final estimate: PPL = 7.9927 +/- 0.05243 | 468s
Q8_0 | 2.19.605.281 I Final estimate: PPL = 7.9944 +/- 0.05245 | 141s
Q6_K | 2.19.019.790 I Final estimate: PPL = 8.0116 +/- 0.05261 | 139s
Q5_K_M | 2.11.041.962 I Final estimate: PPL = 8.0391 +/- 0.05276 | 132s
Q4_K_M | 2.04.434.905 I Final estimate: PPL = 8.1785 +/- 0.05387 | 125s
Q3_K_M | 2.09.201.158 I Final estimate: PPL = 8.5159 +/- 0.05617 | 130s
Q2_K | 2.09.594.193 I Final estimate: PPL = 10.5002 +/- 0.07076 | 130s
